In [97]:
import numpy as np

X = np.load('mnist_test_seq.npy')
X = X[:, :, np.newaxis, :, :]
X.shape

(20, 10000, 1, 64, 64)

In [98]:
X[0, 0, :, :]

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(1, 64, 64), dtype=uint8)

In [99]:
from torch.utils.data import DataLoader, Dataset, random_split
import torch
X = torch.from_numpy(X).float()
class MovingObjectsDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return self.X.shape[1]

    def __getitem__(self, idx):
        return self.create_shifted_frames(idx)

    def create_shifted_frames(self, idx):
        #print(self.X.shape)
        sample = self.X[:, idx, :, :]
        return sample

dataset = MovingObjectsDataset(X)
dataloader = DataLoader(dataset, batch_size=1)

#next(iter(dataloader)).size()

In [100]:
tt_split = 0.8
train_dataset, test_dataset = random_split(
    dataset,
    [int(len(dataset) *tt_split), len(dataset) - int(len(dataset) * tt_split)])

train_dataloader = DataLoader(train_dataset, batch_size=32)
test_dataloader = DataLoader(test_dataset)

next(iter(train_dataloader)).size()

torch.Size([32, 20, 1, 64, 64])

In [101]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [102]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2
        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size,
            padding=padding
        )

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)

        i, f, o, g = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

In [103]:
class MovingMNISTPredictor(nn.Module):
    def __init__(self, input_channels=1, hidden_channels=64):
        super().__init__()

        self.hidden_channels = hidden_channels

        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.convlstm = ConvLSTMCell(
            input_channels=hidden_channels,
            hidden_channels=hidden_channels
        )

        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, input_channels, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, future_steps=10):
        """
        x: [B, T, C, H, W]
        returns: [B, future_steps, C, H, W]
        """

        B, T, C, H, W = x.shape
        device = x.device

        h = torch.zeros(B, self.hidden_channels, H, W, device=device)
        c = torch.zeros(B, self.hidden_channels, H, W, device=device)

        # Encode observed frames
        for t in range(T):
            z = self.encoder(x[:, t])
            h, c = self.convlstm(z, h, c)

        predictions = []

        # Autoregressive future prediction
        current_frame = x[:, -1]

        for _ in range(future_steps):
            z = self.encoder(current_frame)
            h, c = self.convlstm(z, h, c)

            pred_frame = self.decoder(h)
            predictions.append(pred_frame)

            current_frame = pred_frame

        return torch.stack(predictions, dim=1)

In [ ]:
from tqdm import tqdm
device = "mps" if torch.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

model = MovingMNISTPredictor().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_dataloader):
        # If dataset returns (video, label), use batch = batch[0]
        if isinstance(batch, (tuple, list)):
            batch = batch[0]
        #batch = torch.unsqueeze(batch, dim=2)
        batch = batch.float().to(device)

        # Normalize if pixels are 0-255
        if batch.max() > 1:
            batch = batch / 255.0

        input_frames = batch[:, :10]   # [B, 10, 1, 64, 64]
        target_frames = batch[:, 10:]  # [B, 10, 1, 64, 64]

        pred_frames = model(input_frames, future_steps=10)

        loss = loss_fn(pred_frames, target_frames)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

 47%|████▋     | 118/250 [06:43<07:31,  3.42s/it]